# MarketAux API Test Notebook

Test and explore the MarketAux fetcher implementation.

In [1]:
import sys
sys.path.insert(0, '/Users/rishimurudkar/ai-paper-trader')

from fetchers import marketaux
import pandas as pd
import logging

# Enable logging to see debug messages
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

## Test 1: Fetch news for specific tickers

In [2]:
articles = marketaux.fetch_news(['AAPL', 'MSFT'], max_results=10)
print(f"Returned {len(articles)} articles\n")

if articles:
    df = pd.DataFrame(articles)
    print(df.to_string())
else:
    print("No articles returned")

INFO: Making MarketAux API request for tickers: ['AAPL', 'MSFT']
INFO: MarketAux API returned 3 articles


Returned 3 articles

                                                                              title                                                                                                                                                                snippet                                                                                                   url                 published_at     source ticker  sentiment_score
0  How Much Would You Have if You Had Invested $2,000 In Apple When It Went Public?  As difficult as this might be to digest, technology titan Apple (NASDAQ: AAPL) is now 50 years old. It reached the milestone on April 1, 2026.\n\nAnd what a run i...              https://finance.yahoo.com/markets/stocks/articles/much-had-invested-2-000-183300927.html  2026-04-04T18:33:00.000000Z  marketaux   AAPL         0.339375
1               Jim Cramer Thinks New Apple (AAPL) CFO Is Settling Well In His Role  Apple Inc. (NASDAQ:AAPL) is one of the\n\n10 Stocks Jim Cram

## Test 2: Fetch with default watchlist (tickers=None)

In [ ]:
# articles = marketaux.fetch_news(None, max_results=10)
print(f"Returned {len(articles)} articles with default watchlist\n")

if articles:
    df = pd.DataFrame(articles)
    print(df.to_string())
    print(f"\nUnique tickers in response: {sorted(df['ticker'].unique())}")
else:
    print("No articles returned")

## Test 3: Inspect a single article structure

In [3]:
# articles = marketaux.fetch_news(['AAPL'], max_results=5)

if articles:
    article = articles[0]
    print("Single Article Structure:")
    print("="*60)
    for key, value in article.items():
        print(f"{key:20s}: {value}")
    
    print("\nValidation:")
    print("="*60)
    required_keys = {'title', 'ticker', 'sentiment_score', 'snippet', 'url', 'published_at', 'source'}
    has_all = required_keys.issubset(article.keys())
    print(f"Has all required keys: {has_all}")
    print(f"Source is 'marketaux': {article['source'] == 'marketaux'}")
    print(f"Sentiment score in range [-1.0, 1.0]: {-1.0 <= article['sentiment_score'] <= 1.0}")
else:
    print("No articles to inspect")

Single Article Structure:
title               : How Much Would You Have if You Had Invested $2,000 In Apple When It Went Public?
snippet             : As difficult as this might be to digest, technology titan Apple (NASDAQ: AAPL) is now 50 years old. It reached the milestone on April 1, 2026.

And what a run i...
url                 : https://finance.yahoo.com/markets/stocks/articles/much-had-invested-2-000-183300927.html
published_at        : 2026-04-04T18:33:00.000000Z
source              : marketaux
ticker              : AAPL
sentiment_score     : 0.339375

Validation:
Has all required keys: True
Source is 'marketaux': True
Sentiment score in range [-1.0, 1.0]: True


## Test 4: Sentiment score distribution

In [4]:
# articles = marketaux.fetch_news(['AAPL', 'MSFT', 'NVDA', 'GOOGL'], max_results=20)

if articles:
    df = pd.DataFrame(articles)
    print("\nSentiment Score Statistics:")
    print("="*60)
    print(df['sentiment_score'].describe())
    
    print("\nSentiment by Ticker:")
    print("="*60)
    sentiment_by_ticker = df.groupby('ticker')['sentiment_score'].agg(['mean', 'min', 'max', 'count'])
    print(sentiment_by_ticker)
else:
    print("No articles returned")


Sentiment Score Statistics:
count    3.000000
mean     0.441340
std      0.103923
min      0.339375
25%      0.388452
50%      0.437529
75%      0.492323
max      0.547117
Name: sentiment_score, dtype: float64

Sentiment by Ticker:
            mean       min       max  count
ticker                                     
AAPL    0.388452  0.339375  0.437529      2
MSFT    0.547117  0.547117  0.547117      1


## Test 5: Rate limiting check

In [ ]:
print("Making 5 sequential API calls...\n")

for i in range(5):
    articles = marketaux.fetch_news(['AAPL'], max_results=1)
    print(f"Call {i+1}: Got {len(articles)} articles")

print("\n(Check logs above for rate limit warnings when approaching 80 requests)")

## Test 6: Compare sentiment across tickers

In [ ]:
articles = marketaux.fetch_news(['AAPL', 'MSFT', 'TSLA', 'JPM', 'QQQ'], max_results=25)

if articles:
    df = pd.DataFrame(articles)
    print("\nArticles sorted by sentiment score (most positive first):")
    print("="*80)
    display_cols = ['ticker', 'title', 'sentiment_score', 'published_at']
    sorted_df = df[display_cols].sort_values('sentiment_score', ascending=False)
    print(sorted_df.to_string())
else:
    print("No articles returned")

## Test 7: Error handling - missing API key simulation

In [ ]:
import os
from unittest.mock import patch

with patch.dict(os.environ, {'MARKETAUX_API_KEY': ''}):
    articles = marketaux.fetch_news(['AAPL'])
    print(f"With missing API key:")
    print(f"  Returns empty list: {articles == []}")
    print(f"  Expected: True")

## Test 8: Summary and validation

In [ ]:
print("\n" + "="*70)
print("MARKETAUX IMPLEMENTATION VALIDATION")
print("="*70)

articles = marketaux.fetch_news(['AAPL', 'MSFT'], max_results=10)

checks = {
    "Returns a list": isinstance(articles, list),
    "List not empty": len(articles) > 0,
}

if articles:
    article = articles[0]
    checks.update({
        "Each item is dict": isinstance(article, dict),
        "Has 'title'": 'title' in article,
        "Has 'ticker'": 'ticker' in article,
        "Has 'sentiment_score'": 'sentiment_score' in article,
        "Has 'snippet'": 'snippet' in article,
        "Has 'url'": 'url' in article,
        "Has 'published_at'": 'published_at' in article,
        "Has 'source'": 'source' in article,
        "Source is 'marketaux'": article['source'] == 'marketaux',
        "Sentiment is numeric": isinstance(article['sentiment_score'], (int, float)),
        "Sentiment in [-1, 1]": -1.0 <= article['sentiment_score'] <= 1.0,
    })

for check, result in checks.items():
    status = "OK" if result else "FAIL"
    print(f"[{status}] {check}")

all_passed = all(checks.values())
print("\n" + "="*70)
if all_passed:
    print("ALL TESTS PASSED")
else:
    print("SOME TESTS FAILED")
print("="*70)